In [1]:
import torch
import pandas as pd
import json
from transformers import AutoProcessor, VoxtralForConditionalGeneration, BitsAndBytesConfig
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="bitsandbytes")

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "mistralai/Voxtral-Mini-3B-2507"

print("Loading Base Model for Zero-Shot Evaluation...")
processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "left"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # Highly optimized for speed/accuracy
    bnb_4bit_use_double_quant=True, # Saves extra memory at no speed cost
    bnb_4bit_compute_dtype=torch.bfloat16
)
model = VoxtralForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    attn_implementation="sdpa", # Apparently using flash attention 2 destroy the model capacity of generating complete phrases
    device_map=device
)
# Force static KV cache allocation
model.generation_config.cache_implementation = "static"
# Compile the model (this will take a minute on the first batch, but speeds up subsequent inference)
model.forward = torch.compile(model.forward)
# Load your unseen test dataset
df_test = pd.read_csv("./data/combined_multimodal_dataset_test.csv")

Loading Base Model for Zero-Shot Evaluation...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

In [2]:
from torch.utils.data import Dataset, DataLoader

class VoxtralAudioDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = f"./data/synthesized_test/{row['Audio_File']}"
        conversation = [
            {"role": "user", "content": [{"type": "audio", "path": audio_path}]}
        ]
        return conversation, row["User_Command"]

def collate_fn(batch):
    conversations = [item[0] for item in batch]
    commands = [item[1] for item in batch]

    # padding=True dynamically pads only to the longest file in THIS batch, saving VRAM
    inputs = processor.apply_chat_template(
        conversations,
        add_generation_prompt=True, # If true tells model to generate the Assistant response
    )
    return inputs, commands

In [3]:
BATCH_SIZE = 36
dataset = VoxtralAudioDataset(df_test)
# num_workers reads the disk in the background while the GPU processes the current batch
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    num_workers=4,
    collate_fn=collate_fn,
    pin_memory=True # Speeds up CPU-to-GPU memory transfer
)
results = []
print(f"Running Batched Baseline Inference (Batch Size: {BATCH_SIZE})...")
for inputs, commands in tqdm(dataloader, desc="Zero-Shot Evaluation"):
    inputs = inputs.to(device, dtype=torch.bfloat16)
    # Generate the response
    outputs = model.generate(
        **inputs,
        do_sample=True, # Enable to use temp and top_k
        temperature=0.2,
        top_p=0.95,
        max_new_tokens=128 # No need to have long context because the base model fails anyway
    )
    decoded_outputs = processor.batch_decode(
        outputs[:, inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    for command, output in zip(commands, decoded_outputs):
        # Attempt to parse JSON (This will likely fail in the baseline)
        valid_json = False
        try:
            # Crude extraction: look for braces
            json_str = output[output.find("{"):output.rfind("}")+1]
            if json_str: # Ensure string is not empty before parsing
                json.loads(json_str)
                valid_json = True
        except:
            pass

        results.append({
            "Command": command,
            "Base_Model_Output": output.strip(),
            "Valid_JSON": valid_json
        })

# Save for thesis comparison
df_results = pd.DataFrame(results)
df_results.to_csv("./models/baseline_results.csv", index=False)
success_rate = df_results["Valid_JSON"].mean() * 100
print(f"\nBaseline Zero-Shot JSON Success Rate: {success_rate:.2f}%")

Running Batched Baseline Inference (Batch Size: 36)...


Zero-Shot Evaluation:   0%|          | 0/296 [00:00<?, ?it/s]

/home/vito/miniconda3/envs/pyqwen/lib/python3.12/site-packages/torch/_inductor/compile_fx.py:321: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
W0526 14:32:49.252000 2285 site-packages/torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode



Baseline Zero-Shot JSON Success Rate: 0.00%
